---
## Step 1 — Imports & Session Config

In [2]:
import os
import pandas as pd
from pyspark.sql.functions import col, to_date, sum, count, round, when

# Allow Delta schema changes for the whole session
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("Session config set.")
print("Spark version:", spark.version)

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 4, Finished, Available, Finished, False)

Session config set.
Spark version: 3.5.5.5.4.20260403.6


In [3]:
import os

print("/lakehouse/default/Files exists:", os.path.exists("/lakehouse/default/Files"))

for root, dirs, files in os.walk("/lakehouse/default/Files"):
    print(root, "->", files[:5])

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 5, Finished, Available, Finished, False)

/lakehouse/default/Files exists: True
/lakehouse/default/Files -> []
/lakehouse/default/Files/nhl_raw -> ['game.csv', 'game_goalie_stats.csv', 'game_goals.csv', 'game_officials.csv', 'game_penalties.csv']


---
## Step 2 — Locate Dataset

Automatically checks the Kaggle cache from RL's previous run.
If not found, falls back to Lakehouse Files.

In [4]:
# Try cache path first (from RL's previous kagglehub download)
CACHE_PATH    = "/home/trusted-service-user/.cache/kagglehub/datasets/martinellis/nhl-game-data/versions/2"
LAKEHOUSE_PATH = "/lakehouse/default/Files/nhl_raw"

if os.path.exists(CACHE_PATH) and len(os.listdir(CACHE_PATH)) > 0:
    path = CACHE_PATH
    print(f"Using cached dataset: {path}")
elif os.path.exists(LAKEHOUSE_PATH) and len(os.listdir(LAKEHOUSE_PATH)) > 0:
    path = LAKEHOUSE_PATH
    print(f"Using Lakehouse Files: {path}")
else:
    raise FileNotFoundError(
        "Dataset not found in cache or Lakehouse Files.\n"
        "Please upload the NHL CSVs to Files/nhl_raw/ in your Lakehouse."
    )

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
print(f"Found {len(csv_files)} CSV files: {csv_files}")

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 6, Finished, Available, Finished, False)

Using Lakehouse Files: /lakehouse/default/Files/nhl_raw
Found 13 CSV files: ['game.csv', 'game_goalie_stats.csv', 'game_goals.csv', 'game_officials.csv', 'game_penalties.csv', 'game_plays.csv', 'game_plays_players.csv', 'game_scratches.csv', 'game_shifts.csv', 'game_skater_stats.csv', 'game_teams_stats.csv', 'player_info.csv', 'team_info.csv']


---
## Step 3 — Load Raw CSVs into Bronze Tables

In [5]:
for file in csv_files:
    file_path  = f"{path}/{file}"
    table_name = file.replace(".csv", "").lower()

    print(f"Loading {file} -> {table_name}")
    pdf = pd.read_csv(file_path)
    sdf = spark.createDataFrame(pdf)

    sdf.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

    print(f"  Saved {table_name}: {pdf.shape[0]:,} rows")

print("\nAll CSVs loaded successfully.")

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 7, Finished, Available, Finished, False)

Loading game.csv -> game
  Saved game: 26,305 rows
Loading game_goalie_stats.csv -> game_goalie_stats
  Saved game_goals: 148,992 rows
Loading game_officials.csv -> game_officials
  Saved game_officials: 105,853 rows
Loading game_penalties.csv -> game_penalties
  Saved game_penalties: 247,828 rows
Loading game_plays.csv -> game_plays
  Saved game_plays: 5,050,529 rows
Loading game_plays_players.csv -> game_plays_players
  Saved game_plays_players: 7,586,604 rows
Loading game_scratches.csv -> game_scratches
  Saved game_scratches: 137,333 rows
Loading game_shifts.csv -> game_shifts
  Saved game_shifts: 11,882,735 rows
Loading game_skater_stats.csv -> game_skater_stats
  Saved game_skater_stats: 945,830 rows
Loading game_teams_stats.csv -> game_teams_stats
  Saved game_teams_stats: 52,610 rows
Loading player_info.csv -> player_info
  Saved player_info: 3,925 rows
Loading team_info.csv -> team_info
  Saved team_info: 33 rows

All CSVs loaded successfully.


/tmp/ipykernel_7291/4197905672.py:6: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  pdf = pd.read_csv(file_path)


---
## Step 4 — Inspect Tables & Row Counts

In [6]:
for table in spark.sql("SHOW TABLES").select("tableName").collect():
    t = table["tableName"]
    print(f"{t}: {spark.table(t).count():,} rows")

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 8, Finished, Available, Finished, False)

game_plays: 5,050,529 rows
game_goals: 148,992 rows
team_info: 33 rows
game_penalties: 247,828 rows
game_officials: 105,853 rows
game_scratches: 137,333 rows
game_skater_stats: 945,830 rows
player_info: 3,925 rows
game: 26,305 rows
game_teams_stats: 52,610 rows
game_shifts: 11,882,735 rows
game_goalie_stats: 56,656 rows
game_plays_players: 7,586,604 rows
silver_game: 23,735 rows
silver_team: 33 rows
silver_player: 3,925 rows
silver_team_game_stats: 47,470 rows
gold_team_performance: 33 rows
gold_team_performance_named: 33 rows
gold_player_performance: 3,353 rows
gold_game_summary: 23,735 rows
silver_team_game_stats_valid: 47,460 rows
silver_game_goalie_stats: 51,163 rows
silver_game_goals: 133,345 rows
silver_game_officials: 95,188 rows
silver_game_penalties: 229,228 rows
silver_game_scratches: 119,749 rows
silver_game_skater_stats: 853,404 rows
silver_game_teams_stats: 47,470 rows
silver_player_info: 3,925 rows
silver_team_info: 33 rows
silver_game_shifts: 9,900,705 rows
silver_game_p

---
## Step 5 — Preview Key Tables

In [7]:
display(spark.table("game").limit(5))
display(spark.table("team_info").limit(5))
display(spark.table("player_info").limit(5))
display(spark.table("game_teams_stats").limit(5))
display(spark.table("game_skater_stats").limit(5))
display(spark.table("game_plays").limit(5))

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 957912c1-548f-4b8a-9430-bc821cc92889)

SynapseWidget(Synapse.DataFrame, 7e923325-1205-4a44-8217-2d1ba3c3b9ac)

SynapseWidget(Synapse.DataFrame, 64cf786f-7967-4c7a-8f30-ee95476002b1)

SynapseWidget(Synapse.DataFrame, a4de78b7-66ff-4a62-8db1-5068caf6ad8c)

SynapseWidget(Synapse.DataFrame, 3427b976-14c6-4cf4-ab77-8700f62b3272)

SynapseWidget(Synapse.DataFrame, 1a267b6a-2a0d-4370-9476-64aed9c64e5c)

---
## Step 6 — Silver Layer: Clean & Deduplicate

In [8]:
# silver_game
silver_game = (
    spark.table("game")
    .dropDuplicates(["game_id"])
    .withColumn("date_time_gmt", col("date_time_gmt").cast("timestamp"))
    .withColumn("game_date", to_date(col("date_time_gmt")))
)
silver_game.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_game")
print(f"silver_game: {silver_game.count():,} rows")

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 10, Finished, Available, Finished, False)

silver_game: 23,735 rows


In [9]:
# silver_team
silver_team = spark.table("team_info").dropDuplicates(["team_id"])
silver_team.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_team")
print(f"silver_team: {silver_team.count()} rows")

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 11, Finished, Available, Finished, False)

silver_team: 33 rows


In [10]:
# silver_player
silver_player = spark.table("player_info").dropDuplicates(["player_id"])
silver_player.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_player")
print(f"silver_player: {silver_player.count():,} rows")

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 12, Finished, Available, Finished, False)

silver_player: 3,925 rows


In [11]:
# silver_team_game_stats
silver_team_game_stats = (
    spark.table("game_teams_stats").dropDuplicates(["game_id", "team_id"])
    .withColumn("settled_in",
        when(col("settled_in") == "tbc", "SO")
        .otherwise(col("settled_in"))
    )
)
silver_team_game_stats.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_team_game_stats")
print(f"silver_team_game_stats: {silver_team_game_stats.count():,} rows")

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 13, Finished, Available, Finished, False)

silver_team_game_stats: 47,470 rows


In [12]:
# silver_game_skater_stats
silver_game_skater_stats = (
    spark.table("game_skater_stats")
    .dropDuplicates(["game_id", "player_id"])
    .withColumn("shooting_pct",
        round(when(col("shots") > 0, col("goals") / col("shots") * 100).otherwise(0), 2))
    .withColumn("points", col("goals") + col("assists"))
    .withColumn("toi_minutes", round(col("timeOnIce") / 60, 2))
    .na.fill(0, subset=["goals", "assists", "shots", "hits", "blocked",
                        "penaltyMinutes", "giveaways", "takeaways", "plusMinus"])
)
silver_game_skater_stats.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_game_skater_stats")
print(f"silver_game_skater_stats: {silver_game_skater_stats.count():,} rows")

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 14, Finished, Available, Finished, False)

silver_game_skater_stats: 853,404 rows


---
## Step 7 — Data Validation

In [13]:
print("--- Duplicate game IDs ---")
spark.sql("""
    SELECT game_id, COUNT(*) AS count FROM silver_game
    GROUP BY game_id HAVING COUNT(*) > 1
""").show()

print("--- Null checks on silver_game ---")
spark.sql("""
    SELECT
        SUM(CASE WHEN game_id IS NULL THEN 1 ELSE 0 END)      AS null_game_ids,
        SUM(CASE WHEN home_team_id IS NULL THEN 1 ELSE 0 END) AS null_home_teams,
        SUM(CASE WHEN away_team_id IS NULL THEN 1 ELSE 0 END) AS null_away_teams
    FROM silver_game
""").show()

print("--- Orphan team IDs in team stats ---")
spark.sql("""
    SELECT t.team_id, COUNT(*) AS missing_count
    FROM silver_team_game_stats t
    LEFT JOIN silver_team s ON t.team_id = s.team_id
    WHERE s.team_id IS NULL
    GROUP BY t.team_id ORDER BY missing_count DESC
""").show()

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 15, Finished, Available, Finished, False)

--- Duplicate game IDs ---
+-------+-----+
|game_id|count|
+-------+-----+
+-------+-----+

--- Null checks on silver_game ---
+-------------+---------------+---------------+
|null_game_ids|null_home_teams|null_away_teams|
+-------------+---------------+---------------+
|            0|              0|              0|
+-------------+---------------+---------------+

--- Orphan team IDs in team stats ---
+-------+-------------+
|team_id|missing_count|
+-------+-------------+
|     88|            3|
|     87|            3|
|     90|            2|
|     89|            2|
+-------+-------------+



---
## Step 8 — Remove Orphan Records & Create Validated Silver Table

In [14]:
# Inner join removes 10 orphan records (team_ids 87-90 with no team reference)
silver_team_game_stats_valid = (
    spark.table("silver_team_game_stats").alias("t")
    .join(spark.table("silver_team").alias("s"),
          col("t.team_id") == col("s.team_id"), "inner")
    .select("t.*")
)
silver_team_game_stats_valid.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver_team_game_stats_valid")
print(f"silver_team_game_stats_valid: {silver_team_game_stats_valid.count():,} rows")

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 16, Finished, Available, Finished, False)

silver_team_game_stats_valid: 47,460 rows


---
## Step 9 — Gold Layer: Team Performance

In [15]:
gold_team_performance = (
    spark.table("silver_team_game_stats_valid")
    .groupBy("team_id")
    .agg(
        count("game_id").alias("games_played"),
        sum(col("won").cast("int")).alias("wins"),
        sum("goals").alias("goals_for"),
        sum("shots").alias("shots_for"),
        sum("hits").alias("total_hits"),
        sum("pim").alias("penalty_minutes"),
        sum("powerPlayGoals").alias("power_play_goals"),
        sum("powerPlayOpportunities").alias("power_play_opportunities"),
        sum("giveaways").alias("giveaways"),
        sum("takeaways").alias("takeaways")
    )
    .withColumn("losses", col("games_played") - col("wins"))
    .withColumn("win_percentage",
        round((col("wins") / col("games_played")) * 100, 2))
    .withColumn("avg_goals_per_game",
        round(col("goals_for") / col("games_played"), 2))
    .withColumn("avg_shots_per_game",
        round(col("shots_for") / col("games_played"), 2))
    .withColumn("power_play_success_rate",
        round(when(col("power_play_opportunities") > 0,
                   col("power_play_goals") / col("power_play_opportunities") * 100)
              .otherwise(0), 2))
)
gold_team_performance.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_team_performance")
print(f"gold_team_performance: {gold_team_performance.count()} rows")

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 17, Finished, Available, Finished, False)

gold_team_performance: 33 rows


In [16]:
gold_team_performance_named = (
    spark.table("gold_team_performance").alias("g")
    .join(spark.table("silver_team").alias("t"),
          col("g.team_id") == col("t.team_id"), "left")
    .select(
        col("g.team_id"),
        col("t.shortName").alias("team_location"),
        col("t.teamName").alias("team_name"),
        col("t.abbreviation"),
        col("g.games_played"), col("g.wins"), col("g.losses"),
        col("g.goals_for"), col("g.shots_for"), col("g.total_hits"),
        col("g.penalty_minutes"), col("g.power_play_goals"),
        col("g.power_play_opportunities"), col("g.giveaways"), col("g.takeaways"),
        col("g.win_percentage"), col("g.avg_goals_per_game"),
        col("g.avg_shots_per_game"), col("g.power_play_success_rate")
    )
)
gold_team_performance_named.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_team_performance_named")
print(f"gold_team_performance_named: {gold_team_performance_named.count()} rows")
display(gold_team_performance_named.orderBy(col("win_percentage").desc()))

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 18, Finished, Available, Finished, False)

gold_team_performance_named: 33 rows


SynapseWidget(Synapse.DataFrame, a9ef0b39-0c7a-4cce-966e-aa546a413c1c)

---
## Step 10 — Gold Layer: Player Performance

In [17]:
gold_player_performance = (
    spark.table("silver_player").alias("p")
    .join(spark.table("silver_game_skater_stats").alias("s"),
          col("p.player_id") == col("s.player_id"), "inner")
    .groupBy(
        col("p.player_id"), col("p.firstName"), col("p.lastName"),
        col("p.nationality"), col("p.primaryPosition")
    )
    .agg(
        count("s.game_id").alias("games_played"),
        sum("s.goals").alias("goals"),
        sum("s.assists").alias("assists"),
        sum("s.shots").alias("shots"),
        sum("s.hits").alias("hits"),
        sum("s.penaltyMinutes").alias("penalty_minutes"),
        sum("s.powerPlayGoals").alias("power_play_goals"),
        sum("s.powerPlayAssists").alias("power_play_assists"),
        sum("s.takeaways").alias("takeaways"),
        sum("s.giveaways").alias("giveaways"),
        sum("s.blocked").alias("blocked_shots")
    )
    .withColumn("points", col("goals") + col("assists"))
    .withColumn("avg_points_per_game",
        round(col("points") / col("games_played"), 2))
    .withColumn("shot_conversion_rate",
        round(when(col("shots") > 0, col("goals") / col("shots") * 100)
              .otherwise(None), 2))
)
gold_player_performance.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_player_performance")
print(f"gold_player_performance: {gold_player_performance.count():,} rows")
display(gold_player_performance.orderBy(col("points").desc()))

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 19, Finished, Available, Finished, False)

gold_player_performance: 3,353 rows


SynapseWidget(Synapse.DataFrame, 397adacc-4845-4a69-807a-595c2ba1712c)

---
## Step 11 — Gold Layer: Game Summary

In [18]:
gold_game_summary = (
    spark.table("silver_game").alias("g")
    .join(spark.table("silver_team").alias("home"),
          col("g.home_team_id") == col("home.team_id"), "left")
    .join(spark.table("silver_team").alias("away"),
          col("g.away_team_id") == col("away.team_id"), "left")
    .select(
        col("g.game_id"), col("g.season"), col("g.type"),
        col("g.game_date"), col("g.date_time_gmt"),
        col("home.teamName").alias("home_team"),
        col("away.teamName").alias("away_team"),
        col("g.home_goals"), col("g.away_goals"),
        col("g.outcome"), col("g.venue")
    )
)
gold_game_summary.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_game_summary")
print(f"gold_game_summary: {gold_game_summary.count():,} rows")
display(gold_game_summary)

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 20, Finished, Available, Finished, False)

gold_game_summary: 23,735 rows


SynapseWidget(Synapse.DataFrame, f0316adf-57f1-4986-93c8-b5cbc1e144eb)

---
## Step 12 — Confirm All Tables

In [19]:
print("=== Silver Tables ===")
spark.sql("SHOW TABLES LIKE 'silver*'").show(truncate=False)

print("=== Gold Tables ===")
spark.sql("SHOW TABLES LIKE 'gold*'").show(truncate=False)

StatementMeta(, ea07854a-8a03-435e-bcb9-a6769a654fa3, 21, Finished, Available, Finished, False)

=== Silver Tables ===
+-------------+----------------------------+-----------+
|namespace    |tableName                   |isTemporary|
+-------------+----------------------------+-----------+
|nhl_lakehouse|silver_game                 |false      |
|nhl_lakehouse|silver_team                 |false      |
|nhl_lakehouse|silver_player               |false      |
|nhl_lakehouse|silver_team_game_stats      |false      |
|nhl_lakehouse|silver_team_game_stats_valid|false      |
|nhl_lakehouse|silver_game_goalie_stats    |false      |
|nhl_lakehouse|silver_game_goals           |false      |
|nhl_lakehouse|silver_game_officials       |false      |
|nhl_lakehouse|silver_game_penalties       |false      |
|nhl_lakehouse|silver_game_scratches       |false      |
|nhl_lakehouse|silver_game_skater_stats    |false      |
|nhl_lakehouse|silver_game_teams_stats     |false      |
|nhl_lakehouse|silver_player_info          |false      |
|nhl_lakehouse|silver_team_info            |false      |
|nhl_lake

---
## Step 13 — Housekeeping: All-Time Career Stats + Player Photos

Creates `player_career` (one row per player, all seasons combined) and `dim_player_photos` (photo URLs for headshots).
---

In [ ]:
# player_career — one row per player with career totals across all seasons (now includes goalies)
spark.sql("""
    CREATE OR REPLACE TABLE player_career AS
    SELECT
        player_id,
        full_name,
        position,
        nationality,
        SUM(games_played) AS Career_GP,
        SUM(points) AS Career_PTS,
        SUM(goals) AS Career_G,
        SUM(assists) AS Career_A,
        SUM(shots) AS Career_S,
        ROUND(AVG(avg_save_pct), 4) AS Career_SavePct,
        ROUND(AVG(avg_gaa), 2) AS Career_GAA
    FROM gold_player_season_stats
    WHERE position IN ('C', 'LW', 'RW', 'D', 'G')
    GROUP BY player_id, full_name, position, nationality
""")
cnt = spark.table("player_career").count()
print(f"player_career created: {cnt:,} rows")

In [ ]:
# dim_player_photos — add photo_url to player info
spark.sql("""
    CREATE OR REPLACE TABLE dim_player_photos AS
    SELECT *, CONCAT(
        'https://assets.nhle.com/mugs/nhl/latest/',
        CAST(player_id AS STRING),
        '.png'
    ) AS photo_url
    FROM dim_player
""")
cnt = spark.table("dim_player_photos").count()
spark.table("dim_player_photos").select("player_id", "full_name", "photo_url").show(3)
print(f"dim_player_photos created: {cnt:,} rows")

In [ ]:
# Confirm both new tables exist
spark.sql("SHOW TABLES LIKE 'player*'").show(truncate=False)
spark.sql("SHOW TABLES LIKE 'dim_player*'").show(truncate=False)